In [1]:
import sys


sys.path.append('../')

from bunkatopics import Bunka
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import load_dataset
import random

# import umap
from umap.umap_ import UMAP # My personal Umap bugs so I use this one
from sentence_transformers import SentenceTransformer


model_name = "all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=model_name) # We recommend starting with a small model

#Scientific Litterature Data
dataset = load_dataset("CShorten/ML-ArXiv-Papers")["train"]["title"]
raw_docs = random.sample(dataset, 200)


projection_model = UMAP(
                n_components=5,
                random_state=42,
                n_neighbors=30,# I want to optimise the local structure (5 low, 25 high)
                min_dist = 0.3,
                metric = 'cosine') # I don't want to disperse embeddings

# #embedding_model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
# embedding_model = SentenceTransformer(model_name_or_path="Bunka/sentence_transformer_encoder")

projection_model.n_components

5

In [2]:
bunka = Bunka(embedding_model=embedding_model, 
                projection_model=projection_model)  # the language is automatically detected, make sure the embedding model is adapted

In [3]:
bunka.actual_dimensions

5

In [4]:
# Fit Bunka to your text data
bunka.fit(raw_docs)

2025-05-08 10:52:16 - Bunka - INFO - Processing 2953 tokens
2025-05-08 10:52:16 - Bunka - INFO - Embedding documents... (can take varying amounts of time depending on their size)
2025-05-08 10:52:16 - Bunka - INFO - Reducing dimensions to 5 using UMAP
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-05-08 10:52:20 - Bunka - INFO - Creating separate 2D projection for visualization
2025-05-08 10:52:21 - Bunka - INFO - Extracting meaningful terms from documents...
2025-05-08 10:52:21 - Bunka - INFO - We could not find the adapted model, we set to en_core_web_sm (English) as default
100%|██████████| 200/200 [00:01<00:00, 164.49it/s]


In [12]:
bunka.fig_embeddings

In [13]:
len(bunka.docs[0].embedding)
len(bunka.docs[0].nd_embedding)

5

In [14]:
df_topics = bunka.get_topics(n_clusters=10, name_length=5, min_count_terms = 2) # Specify the number of terms to describe each topic

2025-05-08 10:52:45 - Bunka - INFO - There is not enough data to select terms with a minimum occurrence of 2. Setting min_count_terms to 1
2025-05-08 10:52:45 - Bunka - INFO - Computing the topics
2025-05-08 10:52:45 - Bunka - INFO - Using 5-dimensional topic modeling
2025-05-08 10:52:45 - Bunka - INFO - Performing topic modeling using 5-dimensional embeddings
2025-05-08 10:52:45 - Bunka - INFO - Clustering 200 documents in 5D space


In [15]:
bunka.docs[0].x

3.4163833

In [8]:
#1.7774856

In [16]:
bunka.docs[0].nd_embedding

[-0.6865968704223633,
 2.3745949268341064,
 6.964171409606934,
 3.9817094802856445,
 3.157512903213501]

In [17]:
bunka.visualize_topics()

2025-05-08 10:52:55 - Bunka - INFO - Creating the Bunka Map
